# Fine-tune DistilBERT on SST-2

Fine-tunes `distilbert-base-uncased` on SST-2 for sentiment classification. Takes ~3 min on a Colab T4.

Download the output `model_local.zip`, unzip into the project root as `model_local/`, and start the server with `MODEL_PATH=./model_local`.

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.34" "evaluate>=0.4"

## 2. Load SST-2 and tokenize

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_ID = "distilbert-base-uncased"
MAX_LEN = 256

ds = load_dataset("nyu-mll/glue", "sst2")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=MAX_LEN)

ds_tok = ds.map(tokenize, batched=True)
ds_tok = ds_tok.rename_column("label", "labels")
ds_tok = ds_tok.remove_columns(["sentence", "idx"])
print(ds_tok)

## 3. Fine-tune (1 epoch)

In [ ]:
import numpy as np
import evaluate
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2, id2label=id2label, label2id=label2id
)

acc = evaluate.load("accuracy")
def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return acc.compute(predictions=preds, references=labels)

args = TrainingArguments(
    output_dir="./tmp_out",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=200,
    report_to="none",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=metrics,
)

trainer.train()
print(trainer.evaluate())

## 4. Sanity-check predictions

In [ ]:
import torch
model.eval()
samples = [
    "this movie was absolutely fantastic",
    "completely disappointed and frustrated",
    "meh, it was okay",
]
enc = tokenizer(samples, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(model.device)
with torch.inference_mode():
    logits = model(**enc).logits
probs = torch.softmax(logits, dim=-1)
for s, p in zip(samples, probs):
    label = id2label[int(p.argmax())]
    print(f"{label:>8s}  {float(p.max()):.3f}  {s}")

## 5. Save model + zip for download

Unzip the downloaded `model_local.zip` into the project root so the path is `./model_local/`. Then:

```bash
MODEL_PATH=./model_local uv run uvicorn serve.app:app
```

In [ ]:
import shutil
from google.colab import files

OUT = "model_local"
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
shutil.make_archive(OUT, "zip", OUT)
files.download(f"{OUT}.zip")